# Hallucination Spans Data Preparation


In [13]:
from datasets import load_dataset
import sys
from pathlib import Path
from collections import Counter
import json
import pandas as pd


ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / 'scripts').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))

## 1. Source

In [10]:
raw = load_dataset('minpeter/toolace-parsed', 'toolace', split='train')
print(len(raw), 'dialogues')

11072 dialogues


In [11]:
row = raw[0]
msgs = json.loads(row['messages']) if isinstance(row['messages'], str) else row['messages']
tools = json.loads(row['tools']) if isinstance(row['tools'], str) else row['tools']
print(f'{len(tools)} tools, {len(msgs)} messages')
for m in msgs:
    print(f'  [{m.get("role"):9s}] {str(m.get("content") or "")[:60]!r}')

6 tools, 10 messages
  [user     ] "I'm considering investing and I'd like to know what's happen"
  [assistant] ''
  [tool     ] '{"trends": [{"name": "S&P 500", "description": "Standard & P'
  [assistant] 'Here are the top Market Trends in the US right now:\n\n1. **S&'
  [user     ] 'Could you explain what SEC filings are and how they might be'
  [assistant] 'Sure, SEC filings refer to official documents that are submi'
  [user     ] 'So, a couple of years ago, I invested in a tech startup with'
  [assistant] "As an assistant, I'm here to provide advice and guide you in"
  [user     ] 'Could you provide me with the SEC filings for Apple? Also, I'
  [assistant] ''


## 2. Filter

Each row must have: user query, ≥1 tool, tool call, tool response, final answer. Plus output length in [80, 2500] chars and query↔output content-word overlap ≥ 0.10.

In [14]:
from build_from_toolace import normalize_row

outcomes = Counter()
first_valid = None
for i in range(500):
    rec, reason = normalize_row(raw[i], i, 'minpeter/toolace-parsed', min_overlap=0.10)
    outcomes['passed' if rec else reason] += 1
    if rec and first_valid is None: first_valid = rec
pd.DataFrame(outcomes.most_common(), columns=['outcome', 'count'])

,outcome,count
0,passed,447
1,off_topic_output,35
2,output_too_long,9
3,output_too_short,6
4,missing_tool_response,2
5,missing_tool_call,1


In [15]:
print('query: ', first_valid['query'][:100])
print('context:', first_valid['context'][:120] + '…')
print('output: ', first_valid['clean_output'][:120] + '…')

query:  I'm considering investing and I'd like to know what's happening in the market right now. Could you g
context: {"content": {"trends": [{"data": {"current_value": "4172.80", "percentage_change": "+0.68%"}, "description": "Standard &…
output:  As an assistant, I'm here to provide advice and guide you in your financial decisions. Here are my thoughts:

Investing …


## 3. Corrupt

From one clean example produce up to 4 records: `clean`, `contradiction`, `overgeneration`, `missing_tool`.

In [16]:
from corruptors import make_corruptions, collect_corpus_pool

pool = collect_corpus_pool([first_valid])
records = make_corruptions(first_valid, include_clean=True, grounded_pool=pool)
for r in records:
    print(f'{r["meta"]["corruption_type"]:14s} labels={len(r["hallucination_labels"])}')

clean          labels=0
contradiction  labels=1
overgeneration labels=1
missing_tool   labels=1


In [7]:
def show(r):
    out = r['output']
    for lab in sorted(r['hallucination_labels'], key=lambda l: -l['start']):
        out = out[:lab['start']] + '⟦' + out[lab['start']:lab['end']] + '⟧' + out[lab['end']:]
    return out

for r in records:
    print(f'─── {r["meta"]["corruption_type"]} ───')
    print(show(r)[:500])
    print()

─── clean ───
As an assistant, I'm here to provide advice and guide you in your financial decisions. Here are my thoughts:

Investing in startups or any company without conducting proper research carries high risk. Although some startups have very high growth potentials and can provide excellent returns, many also fail within a few years of operation. This is especially the case in the tech industry where competition is stiff and technology changes rapidly.

That being said, your past experience with the tech

─── contradiction ───
As an assistant, I'm here to provide advice and guide you in your financial decisions. Here are my thoughts:

Investing in startups or any company without conducting proper research carries high risk. Although some startups have very high growth potentials and can provide excellent returns, many also fail within a few years of operation. This is especially the case in the tech industry where competition is stiff and technology changes rapidly.

That being sa

## 4. Dataset shape

In [8]:
data = ROOT / 'data'
load = lambda p: [json.loads(l) for l in p.open()]
sizes = []
for cfg in ['combined', 'contradiction', 'missing_tool', 'overgeneration']:
    sizes.append({
        'config': cfg,
        **{s: len(load(data / cfg / f'{s}.jsonl')) for s in ('train', 'validation', 'test')}
    })
pd.DataFrame(sizes)

,config,train,validation,test
0,combined,2118,264,264
1,contradiction,966,120,120
2,missing_tool,1152,144,144
3,overgeneration,1152,144,144


## 6. Load from Hub

In [11]:
from datasets import get_dataset_config_names
REPO = 'Ivan1008/toolace-hallucination-spans'
print(get_dataset_config_names(REPO))
ds = load_dataset(REPO, 'combined')
ds

['combined', 'contradiction', 'missing_tool', 'overgeneration']


DatasetDict({
    train: Dataset({
        features: ['context', 'hallucination_labels', 'id', 'meta', 'output', 'query'],
        num_rows: 2118
    })
    validation: Dataset({
        features: ['context', 'hallucination_labels', 'id', 'meta', 'output', 'query'],
        num_rows: 264
    })
    test: Dataset({
        features: ['context', 'hallucination_labels', 'id', 'meta', 'output', 'query'],
        num_rows: 264
    })
})